# Loan Approval Prediction - Live Demo
**COEN 330 - Applied Machine Learning**

This notebook loads the final trained pipeline and estimates whether a new application resembles **historically approved** or **historically rejected** applications. Based on that, it makes prediction if the new applicant's loan is going to be approved or rejected.

## Step 1 - Load the trained pipeline

In [ ]:
import sys
from pathlib import Path

import joblib
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

MODEL_FILENAME = "GradientBoosting.pkl"


def find_project_root(start: Path | None = None) -> Path:
    """Find the project root regardless of where Jupyter was launched."""
    start = (start or Path.cwd()).resolve()

    for candidate in [start, *start.parents]:
        if (
            (candidate / "models" / MODEL_FILENAME).exists()
            and (candidate / "src").exists()
        ):
            return candidate

    raise FileNotFoundError(
        "Could not find the project root. Expected a folder containing "
        f"'models/{MODEL_FILENAME}' and 'src/'."
    )


PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / "src"
MODEL_PATH = PROJECT_ROOT / "models" / MODEL_FILENAME

# The saved pipeline references custom classes from src/preprocessing.py.
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import preprocessing  # noqa: F401  # required when unpickling the pipeline

pipeline = joblib.load(MODEL_PATH)

print(f"Project root: {PROJECT_ROOT}")
print(f"Loaded model: {MODEL_PATH.name}")
print(f"Pipeline steps: {list(pipeline.named_steps)}")


## Step 2 - Define an applicant

Change the values below and rerun this cell.  
`loan_percent_income` is calculated automatically to keep the inputs internally consistent.


In [ ]:
applicant_a = {
    "person_age": 29,
    "person_gender": "male",
    "person_education": "Bachelor",
    "person_income": 55_000,
    "person_emp_exp": 4,
    "person_home_ownership": "RENT",
    "loan_amnt": 12_000,
    "loan_intent": "EDUCATION",
    "loan_int_rate": 11.5,
    "cb_person_cred_hist_length": 5,
    "credit_score": 640,
    "previous_loan_defaults_on_file": "No",
}

applicant_a["loan_percent_income"] = (
    applicant_a["loan_amnt"] / applicant_a["person_income"]
)

pd.DataFrame.from_dict(
    applicant_a,
    orient="index",
    columns=["Value"],
).rename_axis("Feature")


## Step 3 - Predict

In [ ]:
DEFAULT_THRESHOLD = 0.50


def predict_applicant(applicant: dict, threshold: float = DEFAULT_THRESHOLD) -> dict:
    """Run one applicant through the complete saved preprocessing/model pipeline."""
    X_new = pd.DataFrame([applicant])

    if not hasattr(pipeline, "predict_proba"):
        raise TypeError(
            "This demo expects the selected model to provide predict_proba()."
        )

    probability = float(pipeline.predict_proba(X_new)[0, 1])
    prediction = "Predicted APPROVED" if probability >= threshold else "Predicted REJECTED"

    return {
        "prediction": prediction,
        "approval_probability": probability,
        "threshold": threshold,
    }


result_a = predict_applicant(applicant_a)

print("=" * 58)
print(f"MODEL OUTPUT           : {result_a['prediction']}")
print(f"ESTIMATED P(APPROVED)  : {result_a['approval_probability']:.1%}")
print(f"DECISION THRESHOLD     : {result_a['threshold']:.2f}")
print("=" * 58)
print("This output reflects historical approval patterns; it is not a lending recommendation.")


## Step 4 - Compare with a lower-approval-profile example

This second profile is included only to demonstrate that the model responds to different input values.  
It should not be described as “high risk,” because the dataset does not contain a default outcome.


In [ ]:
applicant_b = {
    "person_age": 22,
    "person_gender": "female",
    "person_education": "High School",
    "person_income": 18_000,
    "person_emp_exp": 0,
    "person_home_ownership": "RENT",
    "loan_amnt": 14_000,
    "loan_intent": "PERSONAL",
    "loan_int_rate": 18.5,
    "cb_person_cred_hist_length": 2,
    "credit_score": 480,
    "previous_loan_defaults_on_file": "Yes",
}

applicant_b["loan_percent_income"] = (
    applicant_b["loan_amnt"] / applicant_b["person_income"]
)

result_b = predict_applicant(applicant_b)

comparison = pd.DataFrame(
    [
        {
            "Profile": "Applicant A",
            "Prediction": result_a["prediction"],
            "P(Approved)": result_a["approval_probability"],
        },
        {
            "Profile": "Applicant B",
            "Prediction": result_b["prediction"],
            "P(Approved)": result_b["approval_probability"],
        },
    ]
)

display(comparison.style.format({"P(Approved)": "{:.1%}"}))


## Step 5 - Visual comparison

In [ ]:
names = comparison["Profile"].tolist()
probabilities = comparison["P(Approved)"].tolist()

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(names, probabilities, width=0.45)

ax.axhline(
    DEFAULT_THRESHOLD,
    linestyle="--",
    linewidth=1,
    label=f"Threshold = {DEFAULT_THRESHOLD:.2f}",
)

for bar, probability in zip(bars, probabilities):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        probability + 0.025,
        f"{probability:.1%}",
        ha="center",
        fontweight="bold",
    )

ax.set_ylim(0, 1.08)
ax.set_ylabel("Estimated P(Approved)")
ax.set_title("Model Predictions for Two Example Applications")
ax.legend()
plt.tight_layout()

plot_dir = PROJECT_ROOT / "results" / "plots"
plot_dir.mkdir(parents=True, exist_ok=True)
plt.savefig(plot_dir / "demo_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
